# Сравнение запусков
Читает локальные `outputs/<run>/train_metrics.jsonl` и `eval-*.json`. Ничего не загружает на HF.

In [ ]:
from pathlib import Path
import os
import sys

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src/config.py").exists())
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config import load_config
CONFIG = os.environ.get("ORTHRUS_CONFIG", "configs/qwen3-0.6b.yaml")
cfg = load_config(CONFIG)
print("Experiment:", cfg.run_name)


In [ ]:
import json
import pandas as pd
import matplotlib.pyplot as plt

RUNS = [cfg.run_name]  # добавь имена других запусков
records, results = [], []
for name in RUNS:
    root = Path(cfg.output_dir) / name
    history = root / "train_metrics.jsonl"
    if history.exists():
        records.extend({"run": name, **json.loads(line)} for line in history.read_text().splitlines() if line)
    for path in sorted(root.glob("eval-*.json")):
        report = json.loads(path.read_text())
        for suite, metrics in report.items():
            if not suite.startswith("_"):
                results.append({"run": name, "file": path.name, "suite": suite, **metrics})
history = pd.DataFrame(records)
benchmarks = pd.DataFrame(results)
print("Metric records:", len(history), "Benchmark rows:", len(benchmarks))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for axis, metric in zip(axes, ["train/loss", "eval/kl", "generate/tpf"]):
    if not history.empty and metric in history:
        for name, part in history.dropna(subset=[metric]).groupby("run"):
            axis.plot(part["step"], part[metric], label=name, marker=".")
        axis.legend()
    axis.set(title=metric, xlabel="optimizer step")
    axis.grid(alpha=0.2)
plt.tight_layout()
plt.show()


In [ ]:
if not benchmarks.empty:
    columns = [c for c in ["run", "file", "suite", "tpf", "acceptance_length", "tokens_per_second", "speedup", "ar_match"] if c in benchmarks]
    display(benchmarks[columns])
else:
    print("Run scripts/benchmark.sh first.")
